# Running New Models (Stage-1)

Fresh Colab notebook for trying additional Stage-1 models via **Option B (repo-direct)**.

Scope now:
- RD4AD smoke + quick embedding-based anomaly baseline
- Uses existing pilot manifest from Drive
- Produces CSV outputs for comparison

In [ ]:
# Cell 1: Mount drive + clone/pull repo
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import json
from pathlib import Path

if not Path('/content/FYP-code').exists():
    !git clone https://github.com/spinelessknave8/FYP_code.git /content/FYP-code

%cd /content/FYP-code
!git fetch origin
!git reset --hard origin/main

BASE = Path('/content/drive/MyDrive/fyp_outputs/severstral_compare_small/consistent_pilot')
MANIFEST = BASE / 'pilot_manifest.json'
assert MANIFEST.exists(), f"Missing {MANIFEST}"
manifest = json.loads(MANIFEST.read_text())

print('Loaded manifest:', MANIFEST)
print('normal_train:', len(manifest['normal_train']))
print('normal_test :', len(manifest['normal_test']))
print('known_test  :', len(manifest['known_test']))
print('unknown_test:', len(manifest['unknown_test']))

In [ ]:
# Cell 2: Install RD4AD deps + clone repo
import sys, subprocess
from pathlib import Path

subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
    'torch==2.2.2',
    'torchvision==0.17.2',
    'timm==0.9.7',
    'numpy==1.26.4',
    'scikit-learn==1.3.2',
    'opencv-python-headless==4.10.0.84',
    'tqdm',
    'pyyaml',
])

if not Path('/content/RD4AD').exists():
    !git clone https://github.com/hq-deng/RD4AD /content/RD4AD
else:
    %cd /content/RD4AD
    !git pull --ff-only
    %cd /content/FYP-code

print('RD4AD repo ready.')

In [ ]:
# Cell 3: RD4AD smoke test
%cd /content/RD4AD
%%writefile /tmp/smoke_rd4ad.py
import torch
from resnet import resnet18
from de_resnet import de_resnet18

enc, bn = resnet18(pretrained=False)
dec = de_resnet18(pretrained=False)
enc.eval(); bn.eval(); dec.eval()

x = torch.randn(2, 3, 256, 256)
with torch.no_grad():
    feats = enc(x)
    z = bn(feats)
    out = dec(z)

print('RD4AD smoke OK')
print('encoder feats:', [f.shape for f in feats])
print('decoder out:', [o.shape for o in out])

!python /tmp/smoke_rd4ad.py
%cd /content/FYP-code

In [ ]:
# Cell 4: Build tiny RD4AD-format data from pilot manifest
from pathlib import Path
import os, shutil, json

BASE = Path('/content/drive/MyDrive/fyp_outputs/severstral_compare_small/consistent_pilot')
manifest = json.loads((BASE / 'pilot_manifest.json').read_text())

out = BASE / 'rd4ad_data'
if out.exists():
    shutil.rmtree(out)

for p in [
    out / 'train' / 'good',
    out / 'test' / 'good',
    out / 'test' / 'known_defect',
    out / 'test' / 'unknown_defect',
]:
    p.mkdir(parents=True, exist_ok=True)

def symlink_many(paths, target_dir):
    for i, src in enumerate(paths):
        srcp = Path(src)
        if not srcp.exists():
            continue
        dst = target_dir / f"{i:06d}_{srcp.name}"
        try:
            os.symlink(srcp, dst)
        except:
            shutil.copy2(srcp, dst)

symlink_many(manifest['normal_train'], out / 'train' / 'good')
symlink_many(manifest['normal_test'], out / 'test' / 'good')
symlink_many([p for p,_ in manifest['known_test']], out / 'test' / 'known_defect')
symlink_many([p for p,_ in manifest['unknown_test']], out / 'test' / 'unknown_defect')

print('RD4AD data root:', out)

In [ ]:
# Cell 5: RD4AD encoder + kNN anomaly baseline (Stage-1 quick result)
import torch, numpy as np, json
from pathlib import Path
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from sklearn.metrics import roc_auc_score, roc_curve
from sklearn.neighbors import NearestNeighbors
import pandas as pd
import sys

sys.path.insert(0, '/content/RD4AD')
from resnet import resnet18

device = 'cuda' if torch.cuda.is_available() else 'cpu'

enc, _ = resnet18(pretrained=True)
enc = enc.to(device).eval()

tf = transforms.Compose([
    transforms.Resize((256,256)),
    transforms.Grayscale(num_output_channels=3),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
])

class DS(Dataset):
    def __init__(self, paths): self.paths = paths
    def __len__(self): return len(self.paths)
    def __getitem__(self, i):
        x = tf(Image.open(self.paths[i]).convert('RGB'))
        return x

BASE = Path('/content/drive/MyDrive/fyp_outputs/severstral_compare_small/consistent_pilot')
manifest = json.loads((BASE / 'pilot_manifest.json').read_text())

norm_train = manifest['normal_train']
norm_test = manifest['normal_test']
known_test = [p for p,_ in manifest['known_test']]
unk_test = [p for p,_ in manifest['unknown_test']]
def_test = known_test + unk_test

def embed(paths, bs=32):
    dl = DataLoader(DS(paths), batch_size=bs, shuffle=False, num_workers=2)
    out = []
    with torch.no_grad():
        for x in dl:
            x = x.to(device)
            feats = enc(x)
            z = feats[-1].mean(dim=(2,3))
            out.append(z.cpu().numpy())
    return np.concatenate(out, axis=0)

E_ntr = embed(norm_train)
E_nts = embed(norm_test)
E_kts = embed(known_test)
E_uts = embed(unk_test)
E_dts = np.concatenate([E_kts, E_uts], axis=0)

nn = NearestNeighbors(n_neighbors=5, metric='euclidean').fit(E_ntr)
def score(E):
    d, _ = nn.kneighbors(E)
    return d.mean(axis=1)

s_n = score(E_nts)
s_k = score(E_kts)
s_u = score(E_uts)
s_d = np.concatenate([s_k, s_u])

y = np.concatenate([np.zeros(len(s_n)), np.ones(len(s_d))])
s = np.concatenate([s_n, s_d])

auc = roc_auc_score(y, s)
fpr, tpr, thr = roc_curve(y, s)

def op(cap):
    i = int(np.argmin(np.abs(fpr - cap)))
    tau = thr[i]
    return {
        'fpr_target': cap,
        'tau': float(tau),
        'fpr_normal': float(np.mean(s_n > tau)),
        'tpr_defect': float(np.mean(s_d > tau)),
        'tpr_unknown': float(np.mean(s_u > tau)),
        'fpr_known_as_def': float(np.mean(s_k > tau)),
    }

ops = [op(0.05), op(0.10), op(0.20)]

print('RD4AD-encoder kNN AUROC(normal-vs-defect):', round(float(auc), 4))
print('Operating points:', ops)

rows = [{'model':'rd4ad_encoder_knn', 'auroc_defect_screening': float(auc), **o} for o in ops]
df = pd.DataFrame(rows)
display(df)

out_csv = BASE / 'rd4ad_quick_results.csv'
df.to_csv(out_csv, index=False)
print('Wrote:', out_csv)

## Next

After this runs, add SimpleNet and CFLOW cells in this same notebook.

Keep outputs in:
- `.../rd4ad_quick_results.csv`
- `.../simplenet_quick_results.csv` (to add)
- `.../cflow_quick_results.csv` (to add)